# Search Lab — BFS and A* Warehouse Navigation

**Artificial Intelligence Laboratory: Search and A\***  
**Topic:** Using an LLM as an Engineering Assistant

## Aim

Find a shortest collision-free path from `S` to `G` in the supplied warehouse map, compare BFS with A*, investigate heuristic choices, and evaluate the LLM-assisted implementation.

**Source task:** `search_lab_ex.pdf`


## Task 0 — Formulate the search problem

| Component | Warehouse specification |
| --- | --- |
| States `S` | Traversable grid coordinates `(row, column)` |
| Actions `A` | Up, Down, Left, Right |
| Transition `T` | Move exactly one square if it remains in bounds and is not `#` |
| Initial state `s0` | Coordinate containing `S` |
| Goal `G` | Coordinate containing `G` |
| Cost `c` | 1 per move |

A state needs only the robot's location because the map is fixed. An action is invalid if it leaves the map or enters an obstacle. This is deterministic: each valid action has one known successor. A solution is a sequence of valid moves from `S` to `G`.

## Task 1 — Agent design

The map is a sequence of ASCII rows. The frontier stores locations; a `parent` map both marks discovered states and reconstructs the path. A* additionally stores the best `g` cost and prioritises its frontier by `f = g + h`. On termination, the agent reports found status, directions, length, and states expanded.


## Task 2 — Implement A*

### Question from the PDF

Implement A* using Manhattan distance `h(n)=|x−xG|+|y−yG|`, avoiding unnecessary repeated exploration and reconstructing the final route.

### Code guide

`locate_symbols` defines the start and goal. `neighbours` implements legal actions and transitions. `astar` uses a priority queue; its heap entries are `(f, insertion_order, state)`. Manhattan distance is appropriate because each orthogonal move can reduce the row/column difference by at most one. `bfs` is included for the required comparison.


In [2]:
"""A* and BFS experiments for the warehouse search laboratory.

Run all required experiments with:

    python3 search_lab.py

Positions are represented as ``(row, column)`` tuples.  The search space is
the set of non-obstacle cells, and every legal movement has cost 1.
"""

from dataclasses import dataclass
from heapq import heappop, heappush
from itertools import count
from math import sqrt
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple


Grid = Sequence[str]
Position = Tuple[int, int]
Heuristic = Callable[[Position, Position], float]

ORIGINAL_WAREHOUSE: Grid = (
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################",
)

TRIVIAL: Grid = ("#####", "#SG##", "#####")
NO_SOLUTION: Grid = (
    "#######",
    "#S....#",
    "###.###",
    "#...#G#",
    "#######",
)

# There are two routes: the direct upper route is 4 moves; the lower route is 6.
ALTERNATIVE: Grid = (
    "#######",
    "#S...G#",
    "#.....#",
    "#######",
)

MOVES: Tuple[Tuple[str, int, int], ...] = (
    ("Up", -1, 0),
    ("Down", 1, 0),
    ("Left", 0, -1),
    ("Right", 0, 1),
)


@dataclass
class SearchResult:
    """The observable result of one search run."""

    path: Optional[List[Position]]
    states_expanded: int

    @property
    def found(self) -> bool:
        return self.path is not None

    @property
    def path_length(self) -> Optional[int]:
        return None if self.path is None else len(self.path) - 1


def locate_symbols(grid: Grid) -> Tuple[Position, Position]:
    """Validate a rectangular grid and return its unique S and G positions."""

    if not grid or not grid[0] or any(len(row) != len(grid[0]) for row in grid):
        raise ValueError("The grid must be non-empty and rectangular.")
    allowed = {"#", ".", "S", "G"}
    if any(cell not in allowed for row in grid for cell in row):
        raise ValueError("The grid contains a symbol outside #, ., S, and G.")
    starts = [(r, c) for r, row in enumerate(grid) for c, cell in enumerate(row) if cell == "S"]
    goals = [(r, c) for r, row in enumerate(grid) for c, cell in enumerate(row) if cell == "G"]
    if len(starts) != 1 or len(goals) != 1:
        raise ValueError("The grid must contain exactly one S and one G.")
    return starts[0], goals[0]


def neighbours(position: Position, grid: Grid) -> Iterable[Tuple[Position, str]]:
    """Yield each reachable neighbour and the action that reaches it."""

    row, column = position
    for action, dr, dc in MOVES:
        new_position = (row + dr, column + dc)
        r, c = new_position
        if 0 <= r < len(grid) and 0 <= c < len(grid[0]) and grid[r][c] != "#":
            yield new_position, action


def reconstruct_path(parent: Dict[Position, Optional[Position]], goal: Position) -> List[Position]:
    """Follow parent pointers from goal to start and reverse the result."""

    path: List[Position] = []
    current: Optional[Position] = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    return list(reversed(path))


def bfs(grid: Grid) -> SearchResult:
    """Run breadth-first search; useful as the uninformed baseline."""

    start, goal = locate_symbols(grid)
    frontier = [start]
    parent: Dict[Position, Optional[Position]] = {start: None}
    head = 0
    expanded = 0
    while head < len(frontier):
        current = frontier[head]
        head += 1
        if current == goal:
            return SearchResult(reconstruct_path(parent, goal), expanded)
        expanded += 1
        for next_position, _action in neighbours(current, grid):
            if next_position not in parent:
                parent[next_position] = current
                frontier.append(next_position)
    return SearchResult(None, expanded)


def astar(grid: Grid, heuristic: Heuristic) -> SearchResult:
    """Run A* using f(n) = g(n) + h(n).

    ``g_score`` stores the best known cost to each state.  The heap stores
    (f-score, insertion-order, state), giving deterministic tie handling.
    Stale heap entries are ignored when a better route has since been found.
    """

    start, goal = locate_symbols(grid)
    sequence = count()
    g_score: Dict[Position, int] = {start: 0}
    parent: Dict[Position, Optional[Position]] = {start: None}
    frontier: List[Tuple[float, int, Position]] = [(heuristic(start, goal), next(sequence), start)]
    expanded = 0

    while frontier:
        f_score, _order, current = heappop(frontier)
        expected_f = g_score[current] + heuristic(current, goal)
        if f_score != expected_f:
            continue  # stale entry
        if current == goal:
            return SearchResult(reconstruct_path(parent, goal), expanded)
        expanded += 1

        for next_position, _action in neighbours(current, grid):
            tentative_g = g_score[current] + 1
            if tentative_g < g_score.get(next_position, float("inf")):
                g_score[next_position] = tentative_g
                parent[next_position] = current
                f_next = tentative_g + heuristic(next_position, goal)
                heappush(frontier, (f_next, next(sequence), next_position))
    return SearchResult(None, expanded)


def manhattan(position: Position, goal: Position) -> float:
    """Exact distance if there were no obstacles; admissible here."""

    return abs(position[0] - goal[0]) + abs(position[1] - goal[1])


def euclidean(position: Position, goal: Position) -> float:
    """Straight-line distance, also admissible for four-direction movement."""

    return sqrt((position[0] - goal[0]) ** 2 + (position[1] - goal[1]) ** 2)


def zero(position: Position, goal: Position) -> float:
    """Zero heuristic: A* becomes uniform-cost search."""

    return 0.0


def doubled_manhattan(position: Position, goal: Position) -> float:
    """An intentionally aggressive, generally non-admissible heuristic."""

    return 2 * manhattan(position, goal)


def directions(path: Optional[List[Position]]) -> str:
    """Format a path as human-readable movement actions."""

    if path is None:
        return "No path"
    result = []
    for first, second in zip(path, path[1:]):
        delta = (second[0] - first[0], second[1] - first[1])
        result.append({(-1, 0): "Up", (1, 0): "Down", (0, -1): "Left", (0, 1): "Right"}[delta])
    return " -> ".join(result) if result else "Already at goal"


def print_result(label: str, result: SearchResult) -> None:
    """Print the required measures for one experiment."""

    print(f"{label}: found={result.found}, length={result.path_length}, expanded={result.states_expanded}")
    print(f"  path: {directions(result.path)}")


def main() -> None:
    """Run the original problem, required tests, comparison, and heuristic study."""

    print("=== Original warehouse ===")
    for name, result in (
        ("BFS", bfs(ORIGINAL_WAREHOUSE)),
        ("A* Manhattan", astar(ORIGINAL_WAREHOUSE, manhattan)),
    ):
        print_result(name, result)

    print("\n=== Required tests ===")
    for name, grid, solver in (
        ("Test 1 - original", ORIGINAL_WAREHOUSE, lambda g: astar(g, manhattan)),
        ("Test 2 - trivial", TRIVIAL, lambda g: astar(g, manhattan)),
        ("Test 3 - no solution", NO_SOLUTION, lambda g: astar(g, manhattan)),
        ("Test 4 - alternative paths", ALTERNATIVE, lambda g: astar(g, manhattan)),
    ):
        print_result(name, solver(grid))

    print("\n=== Heuristic investigation on original warehouse ===")
    for name, heuristic in (
        ("h(n) = Manhattan", manhattan),
        ("h(n) = 0", zero),
        ("h(n) = Euclidean", euclidean),
        ("h(n) = 2 * Manhattan", doubled_manhattan),
    ):
        print_result(name, astar(ORIGINAL_WAREHOUSE, heuristic))


if __name__ == "__main__":
    main()


=== Original warehouse ===
BFS: found=True, length=40, expanded=63
  path: Right -> Right -> Right -> Right -> Down -> Down -> Down -> Down -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Up -> Up -> Left -> Left -> Left -> Left -> Left -> Left -> Up -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Down -> Down -> Down -> Down -> Down -> Down
A* Manhattan: found=True, length=40, expanded=63
  path: Right -> Right -> Right -> Right -> Down -> Down -> Down -> Down -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Up -> Up -> Left -> Left -> Left -> Left -> Left -> Left -> Up -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Down -> Down -> Down -> Down -> Down -> Down

=== Required tests ===
Test 1 - original: found=True, length=40, expanded=63
  path: Right -> Right -> Right -> Right -> Down -> Down -> Down -> Down -> Right -> Right -> Right -> Right -> Right -> Right -> Ri

## Task 3 — Systematic tests

Run `main()` above to reproduce the following measured results.

| Test | Found? | Path length | States expanded | Interpretation |
| --- | ---: | ---: | ---: | --- |
| Original warehouse | Yes | 40 | 63 | A valid route reaches `G`. |
| Trivial (`S` adjacent to `G`) | Yes | 1 | 1 | Confirms the one-step base case. |
| No solution | No | — | 9 | Terminates cleanly without looping. |
| Alternative paths | Yes | 4 | 4 | Returns a shortest path. |

The expected directions are printed by the program, so the exact route remains reproducible rather than copied into static prose.


## Task 4 — Inspect A*

| PDF concept | Location in the implementation |
| --- | --- |
| State | `Position = Tuple[int, int]` |
| Action / transition | `MOVES` and `neighbours` |
| Goal test | `if current == goal` |
| `g(n)` | `g_score` |
| `h(n)` | the passed `heuristic`, e.g. `manhattan` |
| `f(n)` | `tentative_g + heuristic(...)` |
| Frontier | `frontier` heap via `heapq` |
| Visited / repeated work | `g_score` and stale-entry test |
| Path reconstruction | `parent` and `reconstruct_path` |

The priority queue selects the lowest `f` value next. The insertion counter makes tied choices deterministic.

**Task 4 answers.**

- **(a)** The A* frontier is a min-heap (`frontier`) implemented with Python's `heapq`.
- **(b)** `heappop(frontier)` selects the entry with the lowest `f(n)`; the insertion order resolves equal priorities deterministically.
- **(c)** The heuristic is evaluated in `heuristic(current, goal)` and when calculating `f_next` for a successor.
- **(d)** Yes. The expression `tentative_g + heuristic(next_position, goal)` explicitly calculates `f(n) = g(n) + h(n)`.
- **(e)** `g_score` only accepts a successor when its new path cost is lower than the best recorded cost; stale heap entries are then skipped.


## Task 5 — Compare A* with blind search

| Measure, original warehouse | BFS | A* with Manhattan |
| --- | ---: | ---: |
| Solution found | Yes | Yes |
| Path length | 40 | 40 |
| States expanded | 63 | 63 |

These are measured outputs from the implementations, not assumed values. The equality is correct for this map: it is a narrow, corridor-like maze, so the only reachable route to the goal forces both methods to expand the same 63 non-goal states. Manhattan distance does not have alternative branches to prune here. Both find equally short paths because movement costs are uniform. On a more open map with several possible routes, A* can expand fewer states by using the heuristic to focus toward the goal.

**Task 5 answers.**

- **(a)** Yes, both BFS and A* find a solution.
- **(b)** Yes, both return a path of length 40.
- **(c)** Neither expands fewer states on this warehouse: both expand 63 states.
- **(d)** A* can expand fewer states when its heuristic ranks promising routes toward the goal ahead of irrelevant alternatives. This map has too little branching for that advantage to appear.

## Task 6 — Heuristic investigation

| Heuristic | Found? | Length | Expanded |
| --- | ---: | ---: | ---: |
| Manhattan | Yes | 40 | 63 |
| `0` (uniform-cost behaviour) | Yes | 40 | 63 |
| Euclidean | Yes | 40 | 63 |
| `2 × Manhattan` | Yes | 40 | 66 |

Manhattan distance is appropriate because the robot may move only horizontally or vertically, each move costs one, and a move can reduce the row or column difference by at most one. It never overestimates the obstacle-free distance to the goal, so it is admissible.

These values were also measured by running the four heuristics on the supplied warehouse. The first three tie for the same structural reason as Task 5: the maze leaves little freedom for the heuristic to change the search order. Manhattan and Euclidean are admissible here; zero provides no directional estimate. Doubling Manhattan is generally non-admissible, so it can sacrifice optimality. It still produces a length-40 route on this map, but it expands 66 states because its inconsistent estimates permit some states to be reopened; the shortest-path guarantee is lost in general.


## Task 7 — Evaluate the LLM-generated agent and final reflection

### Personal responses

The following responses describe the design, implementation, and test evidence recorded in this notebook.

- **Prompt used:** "Implement A* search in Python for an ASCII warehouse grid. Use `(row, column)` states; `S` is the start, `G` is the goal, `#` is blocked, moves are Up/Down/Left/Right, and each move costs 1. Use Manhattan distance, avoid unnecessary repeated exploration, reconstruct the path, and report whether a solution was found, the path length, and states expanded. Also provide BFS for comparison."
- **What I designed myself:** I chose the grid-position state representation, four legal moves, unit movement cost, the goal test, and the required output measures. I also selected the original, adjacent-goal, no-solution, and alternative-route tests before interpreting the results.
- **What the LLM contributed and what I accepted:** The LLM-assisted implementation used a priority queue for the A* frontier, a `g_score` dictionary for the best known cost, parent pointers for path reconstruction, and a BFS baseline. I accepted those structures after checking that they matched the search design.
- **What I changed after testing:** I corrected the inaccurate comment for the alternative-path map: its direct route is 4 moves and lower route is 6, not 8 and 10. I also replaced the unsupported Task 5/6 interpretation with the measured explanation that the supplied warehouse forces a narrow route, which is why BFS, Manhattan A*, zero, and Euclidean all expand 63 states.

### Evaluation of the LLM-generated agent

1. **Correct immediately:** The representation of positions, bounds/obstacle checks, Manhattan heuristic, heap-based frontier, parent-pointer reconstruction, and result reporting all worked on the original warehouse.
2. **Bugs or design problems:** The alternative-map comment claimed incorrect route lengths. The initial tables also needed verification because equal BFS and A* counts can look suspicious without an explanation. The search logic itself produced the recorded valid paths.
3. **How I found them:** I ran all four required tests and then ran BFS and each heuristic on the original grid. The printed paths and counts exposed the comment mismatch and confirmed the table values.
4. **Terminology and data structures:** I checked the roles of the priority queue, `g_score`, stale heap entries, and parent map. The key distinction is that `g_score` records the best cost, while `parent` records the route used to reconstruct the solution.
5. **Modification:** Yes. I corrected the alternative-map documentation and updated the explanations of the measured comparison and heuristic results.
6. **Most useful tests:** The no-solution test was especially important because it checks termination without a path. The alternative-route test checks shortest-path behaviour, while the adjacent-goal test checks the base case.
7. **Could it be trusted without testing?** No. A plausible path does not prove that an algorithm handles blocked cells, repeated states, unreachable goals, or shortest paths correctly.
8. **What I understand about A* now:** A* expands the state with the smallest `f(n) = g(n) + h(n)`. The heuristic changes the expansion order, but it only saves work when the map provides choices it can rank effectively. Admissibility preserves the optimality guarantee; a more aggressive heuristic may alter performance or return a non-optimal route.

### Concise answers to the final reflection questions

1. Formulating the search problem first defines the state, valid actions, transition rules, goal, and cost, so the code can be checked against a clear specification.

2. A* is informed because it uses `h(n)`, an estimate of remaining cost, alongside the known cost `g(n)` to choose what to expand next.

3. The heuristic matters because it controls the search order. An admissible heuristic preserves the shortest-path guarantee, while an uninformative heuristic behaves like uniform-cost search and an over-aggressive one can lose optimality.

4. The LLM accelerated the translation of the design into standard Python structures and gave a starting point for A* and BFS, but it did not replace inspecting the algorithm or interpreting the experimental results.

5. Accepting LLM-generated code without tests could leave unnoticed boundary errors, repeated-state problems, wrong path reconstruction, failure to terminate on unreachable goals, or misleading performance claims. The test results are evidence; a plausible-looking route is not sufficient.
